# Week 7 (W7-2) - Customer behaviour from the order lines
**Owner:** Salim Bello Muhammad (BI Analyst)

**Question.** How do the 500 customers differ in how recently, how often and how much they buy, and are those differences real or the kind of spread chance alone would produce?

**Basis.** Everything is built from the delivered order lines (18,033 orders; cancelled orders set aside), as the Q13 rule requires: the customer master's own `customer_since`, `last_purchase_date` and `total_purchase_value` are not used. Revenue is `line_total` before GST. Region and customer type come from `customers_clean.csv`.

**Limits stated up front.** Order history covers 2019 to 2024 only, and every customer first ordered in 2019 and last ordered in 2024 (Q13). This notebook therefore **describes behaviour and makes no churn, tenure or lifetime claim**. Recency is measured to the end of the order history.

**Method set in advance.** Unit = the customer. Group differences: Kruskal-Wallis with epsilon-squared, Holm-adjusted across all tests. Concentration: Gini and top-share with bootstrap intervals over customers. Chance test: shuffle customer labels across orders (5,000 times) and compare. Seed 42.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
SEED=42; NB=5000; rng=np.random.default_rng(SEED)
def find_dir(env_var, marker, rels):
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
MERGED=find_dir("CADETX_MERGED","sales_orders_full.csv",["merged","data/merged","week-02/reports/week_02/merged","reports/week_02/merged"])
CLEAN=find_dir("CADETX_CLEAN","customers_clean.csv",["clean","data/clean","week-02/reports/week_02/clean","reports/week_02/clean"])
assert MERGED is not None, "Could not find the Week 2 merged folder. Run from inside the repository, or set CADETX_MERGED."
assert CLEAN is not None, "Could not find the Week 2 clean folder (customers_clean.csv). Run from inside the repository, or set CADETX_CLEAN."
OUT=Path("w72_outputs"); OUT.mkdir(exist_ok=True); CH=Path("w72_charts"); CH.mkdir(exist_ok=True)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=.8); ax.set_axisbelow(True)
def holm(p):
    p=np.asarray(p,float); o=np.argsort(p); m=len(p); adj=np.empty(m); run=0
    for k,i in enumerate(o): run=max(run,(m-k)*p[i]); adj[i]=min(1,run)
    return adj
def gini(x):
    x=np.sort(np.asarray(x,float)); n=len(x); return float((2*np.arange(1,n+1)-n-1).dot(x)/(n*x.sum()))
def kw(df,metric,group):
    g=[v[metric].values for _,v in df.groupby(group)]
    H,p=stats.kruskal(*g); n=len(df); k=len(g); e2=max(0.0,(H-k+1)/(n-k)); return H,p,e2,k
S=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,usecols=["so_id","customer_id","order_date","order_status","line_total"],parse_dates=["order_date"])
CU=pd.read_csv(CLEAN/"customers_clean.csv")
dl=S[S.order_status=="Delivered"]

## 1. Customer table from the order lines

In [2]:
O=dl.groupby(["customer_id","so_id"]).agg(order_date=("order_date","first"),revenue=("line_total","sum")).reset_index()
END=O.order_date.max(); START=O.order_date.min()
H=O.groupby("customer_id").agg(orders=("so_id","size"),revenue=("revenue","sum"),first_order=("order_date","min"),last_order=("order_date","max")).reset_index()
H["avg_order_value"]=H.revenue/H.orders
H["recency_days"]=(END-H.last_order).dt.days
H["span_days"]=(H.last_order-H.first_order).dt.days
H["mean_gap_days"]=H.span_days/(H.orders-1).replace(0,np.nan)
def gaps(d):
    s=np.sort(d.values); return np.diff(s).astype("timedelta64[D]").astype(int) if len(s)>1 else np.array([])
G=O.groupby("customer_id").order_date.apply(gaps)
H["median_gap_days"]=H.customer_id.map(G.apply(lambda a: float(np.median(a)) if len(a) else np.nan))
H["longest_gap_days"]=H.customer_id.map(G.apply(lambda a: float(a.max()) if len(a) else np.nan))
mid=pd.Timestamp("2022-01-01")
H["orders_2019_21"]=H.customer_id.map(O[O.order_date<mid].groupby("customer_id").size()).fillna(0).astype(int)
H["orders_2022_24"]=H.customer_id.map(O[O.order_date>=mid].groupby("customer_id").size()).fillna(0).astype(int)
H=H.merge(CU[["customer_id","region","customer_type","industry_segment"]],on="customer_id",how="left")
H.to_csv(OUT/"w72_customer_metrics.csv",index=False)
print(len(H),"customers; history",START.date(),"to",END.date())
check("500 customers, all with delivered orders","500",len(H),len(H)==500)
check("order count equals delivered orders",int(dl.so_id.nunique()),int(H.orders.sum()),int(H.orders.sum())==int(dl.so_id.nunique()))
check("revenue adds up to delivered revenue",int(dl.line_total.sum()),int(H.revenue.sum()),int(H.revenue.sum())==int(dl.line_total.sum()))
check("every customer has a region and a customer type","0 missing",int(H.region.isna().sum()+H.customer_type.isna().sum()),H.region.isna().sum()+H.customer_type.isna().sum()==0)
ok=True
for cid in H.customer_id.sample(50,random_state=1):
    g=dl[dl.customer_id==cid]; r=H[H.customer_id==cid].iloc[0]
    ok&=(g.so_id.nunique()==r.orders) and abs(g.line_total.sum()-r.revenue)<1e-6 and g.order_date.max()==r.last_order
check("50 customers recomputed with a plain filter match","50 of 50","50 of 50" if ok else "mismatch",bool(ok))
check("orders by period add up to orders",int(H.orders.sum()),int(H.orders_2019_21.sum()+H.orders_2022_24.sum()),H.orders.sum()==H.orders_2019_21.sum()+H.orders_2022_24.sum())
check("recency is never negative and ends at the last order date","0 negative; min 0",f"min {H.recency_days.min()}",H.recency_days.min()==0)

500 customers; history 2019-01-01 to 2024-12-31


## 2. How customers differ: recency, frequency, value

In [3]:
cols=["orders","revenue","avg_order_value","recency_days","mean_gap_days","longest_gap_days"]
Q=H[cols].quantile([0,.1,.25,.5,.75,.9,1]).T; Q.columns=["min","p10","p25","median","p75","p90","max"]; Q["mean"]=H[cols].mean()
Q=Q.reset_index().rename(columns={"index":"measure"}); Q.to_csv(OUT/"w72_summary_quartiles.csv",index=False)
print(Q.round(1).to_string(index=False))
print("Customers with a single order:",int((H.orders==1).sum()))
print("Share of customers whose last order is within 90 days of the end:",round((H.recency_days<=90).mean()*100,1),"%; within 365 days:",round((H.recency_days<=365).mean()*100,1),"%")
check("every customer ordered more than once","0 single-order customers",int((H.orders==1).sum()),(H.orders==1).sum()==0)

         measure        min        p10        p25     median        p75        p90        max       mean
          orders       21.0       29.0       32.0       36.0       40.0       45.0       56.0       36.1
         revenue 17721530.0 33441920.0 38446610.0 45321980.0 52438192.5 59173123.0 84966970.0 45917974.9
 avg_order_value   738397.1  1020534.6  1131625.4  1269734.3  1395525.8  1518174.9  1955790.7  1273690.2
    recency_days        0.0        7.0       20.0       46.0       84.0      130.1      358.0       60.9
   mean_gap_days       37.7       47.6       53.5       60.0       66.8       73.8       95.1       60.6
longest_gap_days      115.0      170.9      201.8      240.0      294.2      345.4      532.0      251.8
Customers with a single order: 0
Share of customers whose last order is within 90 days of the end: 77.0 %; within 365 days: 100.0 %


## 3. Concentration of value
Share of revenue from the top 10% and top 20% of customers and the Gini coefficient, each with a 95% bootstrap interval over customers (5,000 resamples).

In [4]:
rev=H.revenue.values; n=len(rev)
def topshare(x,f):
    k=int(round(len(x)*f)); return float(np.sort(x)[::-1][:k].sum()/x.sum())
obs={"top 10% share":topshare(rev,.10),"top 20% share":topshare(rev,.20),"Gini of revenue":gini(rev),"Gini of orders":gini(H.orders.values)}
bs={k:[] for k in obs}
for _ in range(NB):
    ix=rng.integers(0,n,n); r=rev[ix]; o=H.orders.values[ix]
    bs["top 10% share"].append(topshare(r,.10)); bs["top 20% share"].append(topshare(r,.20)); bs["Gini of revenue"].append(gini(r)); bs["Gini of orders"].append(gini(o))
CON=pd.DataFrame([dict(measure=k,value=v,ci_low=np.percentile(bs[k],2.5),ci_high=np.percentile(bs[k],97.5)) for k,v in obs.items()])
CON.to_csv(OUT/"w72_concentration.csv",index=False); print(CON.round(3).to_string(index=False))
# chance test: shuffle customer labels across orders, keeping each order's value
cid=O.customer_id.values; ov=O.revenue.values; codes,uniq=pd.factorize(cid); K=len(uniq)
# (a) revenue spread: shuffle which customer owns each order (order counts per customer stay as they are)
# (b) order-count spread: give every order to a customer drawn at random, all customers equally likely
null_g=[]; null_sd=[]; null_top=[]
for _ in range(NB):
    sh=rng.permutation(codes); r=np.bincount(sh,weights=ov,minlength=K)
    null_g.append(gini(r)); null_top.append(topshare(r,.10))
    c2=np.bincount(rng.integers(0,K,len(codes)),minlength=K); null_sd.append(c2.std(ddof=1))
obs_sd=H.orders.std(ddof=1)
CH_=pd.DataFrame([dict(measure="Gini of revenue",observed=obs["Gini of revenue"],chance_mean=np.mean(null_g),chance_p95=np.percentile(null_g,95),p_value=(1+(np.array(null_g)>=obs["Gini of revenue"]).sum())/(NB+1)),
                  dict(measure="SD of orders per customer",observed=obs_sd,chance_mean=np.mean(null_sd),chance_p95=np.percentile(null_sd,95),p_value=(1+(np.array(null_sd)>=obs_sd).sum())/(NB+1)),
                  dict(measure="Top 10% revenue share",observed=obs["top 10% share"],chance_mean=np.mean(null_top),chance_p95=np.percentile(null_top,95),p_value=(1+(np.array(null_top)>=obs["top 10% share"]).sum())/(NB+1))])
CH_.to_csv(OUT/"w72_chance_test.csv",index=False); print(CH_.round(4).to_string(index=False))
check("top 10% share is between 10% and 100%","10% to 100%",f"{obs['top 10% share']*100:.1f}%",0.10<=obs["top 10% share"]<=1)
check("top 20% share is at least the top 10% share","top20 >= top10",f"{obs['top 20% share']:.3f} >= {obs['top 10% share']:.3f}",obs["top 20% share"]>=obs["top 10% share"])
check("Gini recomputed by pairwise formula matches (50-customer subset)","equal",f"{abs(gini(rev[:50])-(np.abs(rev[:50][:,None]-rev[:50][None,:]).sum()/(2*50*rev[:50].sum()))):.2e}",abs(gini(rev[:50])-(np.abs(rev[:50][:,None]-rev[:50][None,:]).sum()/(2*50*rev[:50].sum())))<1e-9)
check("random assignment of orders gives an order-count spread near the Poisson value","SD about sqrt(mean) = %.2f"%np.sqrt(len(codes)/K),f"{np.mean(null_sd):.2f}",abs(np.mean(null_sd)-np.sqrt(len(codes)/K))<0.3)
check("customer shuffle keeps total revenue and order count","same",f"{int(np.bincount(rng.permutation(codes),weights=ov,minlength=K).sum())}",int(np.bincount(rng.permutation(codes),weights=ov,minlength=K).sum())==int(ov.sum()))

        measure  value  ci_low  ci_high
  top 10% share  0.143   0.139    0.147
  top 20% share  0.266   0.261    0.272
Gini of revenue  0.125   0.117    0.133
 Gini of orders  0.094   0.088    0.099
                  measure  observed  chance_mean  chance_p95  p_value
          Gini of revenue    0.1253       0.1228      0.1281   0.2116
SD of orders per customer    6.0371       6.0054      6.3177   0.4335
    Top 10% revenue share    0.1433       0.1417      0.1444   0.1582


## 4. Do customers differ by region, customer type or industry segment?
Ten Kruskal-Wallis tests (four measures by region and by customer type, plus orders and revenue by industry segment), Holm-adjusted together. Effect size is epsilon-squared (clipped at 0).

In [5]:
tests=[]
for grp in ["region","customer_type"]:
    for m in ["orders","revenue","recency_days","mean_gap_days"]:
        Hs,p,e2,k=kw(H,m,grp); tests.append(dict(group=grp,measure=m,groups=k,H=Hs,p_value=p,effect_size_epsilon2=e2))
for m in ["orders","revenue"]:
    Hs,p,e2,k=kw(H,m,"industry_segment"); tests.append(dict(group="industry_segment",measure=m,groups=k,H=Hs,p_value=p,effect_size_epsilon2=e2))
T=pd.DataFrame(tests); T["holm_p"]=holm(T.p_value.values); T["significant_after_holm"]=T.holm_p<0.05
T.to_csv(OUT/"w72_group_tests.csv",index=False); print(T.round(4).to_string(index=False))
med=[]
for grp in ["region","customer_type","industry_segment"]:
    gg=H.groupby(grp).agg(customers=("customer_id","size"),median_orders=("orders","median"),median_revenue_m=("revenue",lambda x:x.median()/1e6),median_recency_days=("recency_days","median"),median_mean_gap_days=("mean_gap_days","median")).reset_index().rename(columns={grp:"group"}); gg.insert(0,"dimension",grp); med.append(gg)
M=pd.concat(med); M.to_csv(OUT/"w72_group_medians.csv",index=False)
print("Smallest Holm p:",round(T.holm_p.min(),3),"| largest effect size:",round(T.effect_size_epsilon2.max(),3))
check("ten group tests run (four measures by two groupings, plus two by industry)","10",len(T),len(T)==10)
check("Holm-adjusted p is never below the raw p","adj >= raw","ok" if (T.holm_p>=T.p_value-1e-12).all() else "bad",bool((T.holm_p>=T.p_value-1e-12).all()))
check("group medians cover all 500 customers in each dimension","500 each",str(M.groupby('dimension').customers.sum().tolist()),(M.groupby('dimension').customers.sum()==500).all())

           group       measure  groups       H  p_value  effect_size_epsilon2  holm_p  significant_after_holm
          region        orders       6  4.9102   0.4269                0.0000  1.0000                   False
          region       revenue       6  6.3388   0.2746                0.0027  1.0000                   False
          region  recency_days       6  6.2306   0.2844                0.0025  1.0000                   False
          region mean_gap_days       6  6.7324   0.2413                0.0035  1.0000                   False
   customer_type        orders       5  1.8608   0.7613                0.0000  1.0000                   False
   customer_type       revenue       5  3.8153   0.4316                0.0000  1.0000                   False
   customer_type  recency_days       5  2.8968   0.5752                0.0000  1.0000                   False
   customer_type mean_gap_days       5  3.2015   0.5247                0.0000  1.0000                   False
industry_s

## 5. Is a customer's behaviour stable?
If customers truly differ in how often they buy, the customers who ordered a lot in 2019 to 2021 should be the ones who order a lot in 2022 to 2024. Spearman correlation with a permutation p-value; revenue per period is checked the same way. Orders against recency is expected to be negative by construction: a customer with more orders in a fixed window is more likely to have ordered recently.

In [6]:
def perm_rho(x,y):
    obs=stats.spearmanr(x,y)[0]; xr=stats.rankdata(x); yr=stats.rankdata(y)
    d=np.array([np.corrcoef(rng.permutation(xr),yr)[0,1] for _ in range(NB)]); return float(obs),float((1+(np.abs(d)>=abs(obs)).sum())/(NB+1))
rev_a=O[O.order_date<mid].groupby("customer_id").revenue.sum().reindex(H.customer_id).fillna(0).values
rev_b=O[O.order_date>=mid].groupby("customer_id").revenue.sum().reindex(H.customer_id).fillna(0).values
r1,p1=perm_rho(H.orders_2019_21.values,H.orders_2022_24.values); r2,p2=perm_rho(rev_a,rev_b)
r3,p3=perm_rho(H.orders.values,H.avg_order_value.values); r4,p4=perm_rho(H.orders.values,H.recency_days.values)
ST=pd.DataFrame([dict(comparison="Orders 2019-21 vs 2022-24",spearman=r1,p_value=p1),dict(comparison="Revenue 2019-21 vs 2022-24",spearman=r2,p_value=p2),
                 dict(comparison="Orders vs average order value",spearman=r3,p_value=p3),dict(comparison="Orders vs recency (days since last order)",spearman=r4,p_value=p4)])
ST["holm_p"]=holm(ST.p_value.values); ST.to_csv(OUT/"w72_stability_relationships.csv",index=False); print(ST.round(4).to_string(index=False))
check("period order counts are non-negative whole numbers","min >= 0",f"min {int(min(H.orders_2019_21.min(),H.orders_2022_24.min()))}",min(H.orders_2019_21.min(),H.orders_2022_24.min())>=0)
check("period revenue adds up to revenue",int(H.revenue.sum()),int(rev_a.sum()+rev_b.sum()),int(round(rev_a.sum()+rev_b.sum()))==int(H.revenue.sum()))
check("Spearman is within -1 and 1 for every comparison","-1 to 1",f"{ST.spearman.min():.3f} to {ST.spearman.max():.3f}",ST.spearman.abs().max()<=1)

                               comparison  spearman  p_value  holm_p
                Orders 2019-21 vs 2022-24    0.0041   0.9302  1.0000
               Revenue 2019-21 vs 2022-24   -0.0194   0.6631  1.0000
            Orders vs average order value   -0.0037   0.9272  1.0000
Orders vs recency (days since last order)   -0.1351   0.0026  0.0104


## 6. Quiet for longer than usual (descriptive only)
For each customer: is the time since their last order longer than the longest gap they had before? This is shown as a description of the order lines. It is **not a churn measure**: the order history ends at one date, every customer has ordered in 2024, and no customer has left in the data.

In [7]:
H["quiet_longer_than_ever"]=H.recency_days>H.longest_gap_days
Qn=int(H.quiet_longer_than_ever.sum())
print(f"Customers quiet for longer than their longest earlier gap: {Qn} of 500 ({Qn/5:.1f}%)")
print(f"Median longest gap {H.longest_gap_days.median():.0f} days; median recency {H.recency_days.median():.0f} days")
QT=pd.DataFrame([dict(measure="customers quiet longer than their longest earlier gap",count=Qn,share=Qn/500,median_longest_gap_days=H.longest_gap_days.median(),median_recency_days=H.recency_days.median())])
QT.to_csv(OUT/"w72_quiet_descriptive.csv",index=False)
check("quiet count recomputed by loop",str(Qn),str(sum(1 for r in H.itertuples() if r.recency_days>r.longest_gap_days)),Qn==sum(1 for r in H.itertuples() if r.recency_days>r.longest_gap_days))

Customers quiet for longer than their longest earlier gap: 9 of 500 (1.8%)
Median longest gap 240 days; median recency 46 days


## 7. Charts

In [8]:
# 1 Lorenz curve with the chance band
fig,ax=plt.subplots(figsize=(8.2,5.4)); style(ax)
x=np.sort(rev); cum=np.concatenate([[0],np.cumsum(x)/x.sum()]); p=np.linspace(0,1,len(cum))
bandlo=[];bandhi=[]
sims=[]
for _ in range(200):
    sh=rng.permutation(codes); r=np.sort(np.bincount(sh,weights=ov,minlength=K)); sims.append(np.concatenate([[0],np.cumsum(r)/r.sum()]))
sims=np.array(sims); ax.fill_between(p,np.percentile(sims,2.5,axis=0),np.percentile(sims,97.5,axis=0),color=MUTED,alpha=.6,label="Range if customers were shuffled")
ax.plot([0,1],[0,1],color=GRID,lw=1.5); ax.plot(p,cum,color=BLUE,lw=2.5,label="Actual customers")
ax.set_xlabel("Share of customers (smallest to largest revenue)",color=INK2); ax.set_ylabel("Share of revenue",color=INK2)
ax.set_title("Revenue is spread almost as evenly as chance would spread it",loc="left",color=INK,fontsize=13); ax.legend(frameon=False,loc="upper left",labelcolor=INK2)
plt.tight_layout(); plt.savefig(CH/"w72_01_lorenz.png",dpi=150,facecolor=SURF); plt.close()
# 2 orders per customer
fig,ax=plt.subplots(figsize=(8.2,4.8)); style(ax)
ax.hist(H.orders,bins=25,color=BLUE,edgecolor=SURF); ax.axvline(H.orders.median(),color=ORANGE,lw=2)
ax.text(H.orders.median()+.5,ax.get_ylim()[1]*.93,f"median {H.orders.median():.0f} orders",color=INK,fontsize=10)
ax.set_xlabel("Delivered orders per customer, 2019-2024",color=INK2); ax.set_ylabel("Customers",color=INK2); ax.set_title("Most customers order a similar number of times",loc="left",color=INK,fontsize=13)
plt.tight_layout(); plt.savefig(CH/"w72_02_orders_per_customer.png",dpi=150,facecolor=SURF); plt.close()
# 3 stability
fig,ax=plt.subplots(figsize=(8.2,5.0)); style(ax)
ax.scatter(H.orders_2019_21,H.orders_2022_24,s=16,color=BLUE,alpha=.65); ax.set_xlabel("Orders 2019-2021",color=INK2); ax.set_ylabel("Orders 2022-2024",color=INK2)
ax.set_title(f"Past orders do not predict later orders (Spearman {r1:+.2f})",loc="left",color=INK,fontsize=13)
plt.tight_layout(); plt.savefig(CH/"w72_03_stability.png",dpi=150,facecolor=SURF); plt.close()
check("three charts written","3",str(len(list(CH.glob('*.png')))),len(list(CH.glob('*.png')))==3)

## 8. Output files and validation

In [9]:
outs=sorted(p.name for p in OUT.glob("*.csv"))
check("eight result tables written","8",str(len(outs)),len(outs)==8)
V=pd.DataFrame(checks); V.insert(0,"Check_ID",range(1,len(V)+1)); V.to_csv("validation_report_week07_w72.csv",index=False)
print(len(V),"checks,",int((V.Status=="PASS").sum()),"pass"); assert (V.Status=="PASS").all()

22 checks, 22 pass
